# single_neuron_processing
Assesses neuron by neuron data for each recording and then saves into summary files


In [1]:
# imports 
import numpy as np 
import pandas as pd 
import h5py
import h5_utilities_module as h5u
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from pathlib import Path
import matplotlib.colors as mcolors
import warnings
import pingouin as pg
from sklearn.linear_model import LinearRegression
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
import glob
import os
warnings.filterwarnings("ignore", message="Mean of empty slice", category=RuntimeWarning)

In [2]:
# functions
def get_labelled_posteriors(indata, labels):

    '''
    INPUTS:
    indata = posterior probabilites from a classifier with the shape
            n_trials x n_timesteps x n_classes
        
    labels = 1d array with len(n_trials) - these labels ought
            to correspond to class numbers (layers in indata)

    OUTPUT:
        labelled_posteriors = posterior probabilities associated with the
        classes in the labels input for each timestep and trial
    '''

    n_trials, n_times, n_classes = indata.shape
    class_lbls = np.unique(labels)
    class_lbls = class_lbls[~np.isnan(class_lbls)]

    # initialize output
    labelled_posteriors = np.zeros(shape = (n_trials, n_times))

    for ix, lbl in enumerate(class_lbls):
        
        # find trials where this label was chosen
        labelled_posteriors[labels == lbl,:] = indata[labels == lbl,:,int(ix)]
        
    return labelled_posteriors


def pull_balanced_train_set(trials2balance, params2balance):
    '''
    INPUTS:
    trials2balance   - ***logical array*** of the trials you want to balance
    params2balance   - ***list*** where each element is a vector of categorical
                        parameters to balance (e.g. choice value and side)
                        each element of params2balance must have the same
                        number of elements as trials2balance
    OUTPUTS:
    train_ix         - trial indices of a fully balanced training set
    leftover_ix      - trial indices of trials not included in train_ix
    '''

    # Find the indices where trials are selected to balance
    balance_indices = np.where(trials2balance)[0]

    # Create an array of parameters to balance
    params_array = np.array(params2balance).T

    # Find unique combinations and their counts
    p_combos, p_counts = np.unique(params_array[balance_indices], axis=0, return_counts=True)

    # Determine the minimum count for a balanced set
    n_to_keep = np.min(p_counts)

    # Initialize arrays to mark selected and leftover trials
    train_ix = np.zeros(len(trials2balance), dtype=bool)
    leftover_ix = np.zeros(len(trials2balance), dtype=bool)

    # Select a balanced number of trials for each unique parameter combination
    for combo in p_combos:
        # Find indices of trials corresponding to the current combination
        combo_indices = np.where((params_array == combo).all(axis=1) & trials2balance)[0]

        # Shuffle the indices
        np.random.shuffle(combo_indices)

        # Select n_to_keep trials and mark them as part of the training set
        train_ix[combo_indices[:n_to_keep]] = True

        # Mark the remaining trials as leftovers
        leftover_ix[combo_indices[n_to_keep:]] = True

    return train_ix, leftover_ix


def random_prop_of_array(inarray, proportion):
    '''
    INPUTS
    inarray = logical/boolean array of indices to potentially use later
    proportion = how much of inarray should randomly be selected

    OUTPUT
    out_array = logical/boolean that's set as 'true' for a proportion of the 
                initial 'true' values in inarray
    '''

    out_array = np.zeros(shape = (len(inarray), ))

    # find where inarray is true and shuffle those indices
    shuffled_ixs = np.random.permutation(np.asarray(np.where(inarray)).flatten())

    # keep only a proportion of that array
    kept_ix = shuffled_ixs[0: round(len(shuffled_ixs)*proportion)]

    # fill in the kept indices
    out_array[kept_ix] = 1

    # make this a logical/boolean
    out_array = out_array > 0

    return out_array


def pull_from_h5(file_path, data_to_extract):
    try:
        with h5py.File(file_path, 'r') as file:
            # Check if the data_to_extract exists in the HDF5 file
            if data_to_extract in file:
                data = file[data_to_extract][...]  # Extract the data
                return data
            else:
                print(f"'{data_to_extract}' not found in the file.")
                return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None
    
def list_hdf5_data(file_path):
    try:
        with h5py.File(file_path, 'r') as file:
            print(f"Datasets in '{file_path}':")
            for dataset in file:
                print(dataset)
    except Exception as e:
        print(f"An error occurred: {e}")


def get_ch_and_unch_vals(bhv):
    """
    Extracts chosen (ch_val) and unchosen (unch_val) values associated with each trial.

    Parameters:
    - bhv (DataFrame): DataFrame behavioral data.

    Returns:
    - ch_val (ndarray): Array of chosen values for each trial.
    - unch_val (ndarray): Array of unchosen values for each trial. 
                          - places 0s for unchosen values on forced choice trials
    """
    ch_val = np.zeros(shape=(len(bhv, )))
    unch_val = np.zeros(shape=(len(bhv, )))

    bhv['r_val'] = bhv['r_val'].fillna(0)
    bhv['l_val'] = bhv['l_val'].fillna(0)

    ch_left = bhv['side'] == -1
    ch_right = bhv['side'] == 1

    ch_val[ch_left] = bhv['l_val'].loc[ch_left].astype(int)
    ch_val[ch_right] = bhv['r_val'].loc[ch_right].astype(int)

    unch_val[ch_left] = bhv['r_val'].loc[ch_left].astype(int)
    unch_val[ch_right] = bhv['l_val'].loc[ch_right].astype(int)

    return ch_val, unch_val


def get_ch_and_unch_pps(in_pp, bhv, ch_val, unch_val):
    """Gets the posteriors associated with the chosen and unchosen classes

    Args:
        in_pp (ndarray): array of posteriors (n_trials x n_times x n_classes)
        bhv (dataframe): details of each trial
        ch_val (ndarray): vector indicating the class that is ultimately chosen
        unch_val (ndarray): vector indicating the class that was ultimately not chosen

    Returns:
        ch_pp (ndarray): vector of the postior at each point in time for each trial's chosen option
        unch_pp (ndarray): vector of the postior at each point in time for each trial's unchosen option
    """

    # select the chosen and unchosen values 
    n_trials, n_times, n_classes = np.shape(in_pp)
    ch_pp = np.zeros(shape=(n_trials, n_times))
    unch_pp = np.zeros(shape=(n_trials, n_times))

    # loop over each trial
    for t in range(n_trials):
        
        # get the chosen and unchosen PPs
        ch_pp[t, :] = in_pp[t, :, int(ch_val[t]-1)]
        unch_pp[t, :] = in_pp[t, :, int(unch_val[t]-1)]
        
    # set the forced choice unchosen pps to nans, since there was only 1 option
    unch_pp[bhv['forced'] == 1, :] = np.nan
    
    return ch_pp, unch_pp


def get_alt_ch_and_unch_pps(in_pp, bhv, s_ch_val, s_unch_val):
    """Gets the posteriors associated with the chosen and unchosen classes

    Args:
        in_pp (ndarray): array of posteriors (n_trials x n_times x n_classes)
        bhv (dataframe): details of each trial
        s_ch_val (ndarray): vector indicating the class that is ultimately chosen
        s_unch_val (ndarray): vector indicating the class that was ultimately not chosen

    Returns:
        alt_ch_pp (ndarray): vector of the postior at each point in time for the alternative value in the other state
        alt_unch_pp (ndarray): vector of the postior at each point in time for the alternative value in the other state
    """

    # select the chosen and unchosen values 
    n_trials, n_times, n_classes = np.shape(in_pp)
    alt_ch_pp = np.zeros(shape=(n_trials, n_times))
    alt_unch_pp = np.zeros(shape=(n_trials, n_times))

    alt_ch_val = np.zeros_like(s_ch_val)
    alt_unch_val = np.zeros_like(s_unch_val)
    
    alt_ch_val[bhv['state'] == 1] = 8 - s_ch_val[bhv['state'] == 1] + 1
    alt_ch_val[bhv['state'] == 2] = 8 - s_ch_val[bhv['state'] == 2] + 1

    alt_unch_val[bhv['state'] == 1] = 8 - s_unch_val[bhv['state'] == 1] + 1
    alt_unch_val[bhv['state'] == 2] = 8 - s_unch_val[bhv['state'] == 2] + 1

    for t in range(n_trials):
        
        alt_ch_pp[t, :] = in_pp[t, :, int(alt_ch_val[t]-1)]
        alt_unch_pp[t, :] = in_pp[t, :, int(alt_unch_val[t]-1)]

    # set the alternative values to nans for state 3, since there were no alternatives
    alt_ch_pp[bhv['state'] == 3] = np.nan
    alt_unch_pp[bhv['state'] == 3] = np.nan

    return alt_ch_pp, alt_unch_pp

def find_candidate_states(indata, n_classes, temporal_thresh, mag_thresh):
    """Finds periods where decoded posteriors are twice their noise level.

    Args:
        indata (ndarray): 2d array of posterior probabilities associated with some decoder output.
        n_classes (int): How many classes were used in the decoder?
        temporal_thresh (int): Number of contiguous samples that must be above a threshold to be a real state (typically 2).
        mag_thresh (flat): how many times the noise level must a state be? (e.g. 2 = twice the noise level)

    Returns:
        state_details (ndarray): 2d array where each row details when a state occurred [trial_num, time_in_trial, state_length].
        state_array (ndarray): 2d array the same size as indata. It contains 1 in all locations where there were states and 0s everywhere else.
    """
    state_details = np.array([])
    state_array = np.zeros_like(indata)
    
    state_magnitude_thresh = (1 / n_classes) * mag_thresh

    for t in range(indata.shape[0]):
        state_len, state_pos, state_type = find_1dsequences(indata[t, :] > state_magnitude_thresh)
        state_len = state_len[state_type == True]
        state_pos = state_pos[state_type == True]

        for i in range(len(state_len)):
            state_details = np.concatenate((state_details, np.array([t, state_pos[i], state_len[i]])))

    state_details = state_details.reshape(-1, 3)
    state_details = state_details[state_details[:, 2] > temporal_thresh, :]

    # Update state_array using state_details information
    for j in range(len(state_details)):
        state_trial, state_start, state_len = state_details[j].astype(int)
        state_array[state_trial, state_start:(state_start + state_len)] = 1

    return state_details, state_array

def moving_average(x, w, axis=0):
    '''
    Moving average function that operates along specified dimensions of a NumPy array.

    Parameters:
    - x (numpy.ndarray): Input array.
    - w (int): Size of the window to convolve the array with (i.e., smoothness factor).
    - axis (int): Axis along which to perform the moving average (default is 0).

    Returns:
    - numpy.ndarray: Smoothed array along the specified axis with the same size as the input array.
    '''
    x = np.asarray(x)  # Ensure input is a NumPy array
    if np.isnan(x).any():
        x = np.nan_to_num(x)  # Replace NaN values with zeros

    if axis < 0:
        axis += x.ndim  # Adjust negative axis value

    kernel = np.ones(w) / w  # Create kernel for moving average

    # Pad the array before applying convolution
    pad_width = [(0, 0)] * x.ndim  # Initialize padding for each axis
    pad_width[axis] = (w - 1, 0)  # Pad along the specified axis (left side)
    x_padded = np.pad(x, pad_width, mode='constant', constant_values=0)

    # Apply 1D convolution along the specified axis on the padded array
    return np.apply_along_axis(lambda m: np.convolve(m, kernel, mode='valid'), axis, x_padded)

def find_1dsequences(inarray):
        ''' 
        run length encoding. Partial credit to R rle function. 
        Multi datatype arrays catered for including non Numpy
        returns: tuple (runlengths, startpositions, values) 
        '''
        ia = np.asarray(inarray)                # force numpy
        n = len(ia)
        if n == 0: 
            return (None, None, None)
        else:
            y = ia[1:] != ia[:-1]                 # pairwise unequal (string safe)
            i = np.append(np.where(y), n - 1)     # must include last element 
            lens = np.diff(np.append(-1, i))      # run lengths
            pos = np.cumsum(np.append(0, lens))[:-1] # positions
            return(lens, pos, ia[i])
        
        
def calculate_mean_and_interval(data, type='sem', num_samples=1000, alpha=0.05):
    """
    Calculate mean and either SEM or bootstrapped CI for each column of the input array, disregarding NaN values.

    Parameters:
    - data: 2D numpy array
    - type: str, either 'sem' or 'bootstrap_ci'
    - num_samples: int, number of bootstrap samples (applicable only for type='bootstrap_ci')
    - alpha: float, significance level for the confidence interval (applicable only for type='bootstrap_ci')

    Returns:
    - means: 1D numpy array containing means for each column
    - interval: 1D numpy array containing SEMs or bootstrapped CIs for each column
    """
    nan_mask = ~np.isnan(data)
    
    nanmean_result = np.nanmean(data, axis=0)
    n_valid_values = np.sum(nan_mask, axis=0)
    
    if type == 'sem':
        nanstd_result = np.nanstd(data, axis=0)
        interval = nanstd_result / np.sqrt(n_valid_values)
        
    elif type == 'percentile':
        interval = np.mean(np.array([np.abs(nanmean_result - np.nanpercentile (data, 5, axis=0)), np.abs(nanmean_result - np.nanpercentile (data, 95, axis=0))]))
        
        
    elif type == 'bootstrap':
        n_rows, n_cols = data.shape

        # Initialize array to store bootstrap means
        bootstrap_means = np.zeros((num_samples, n_cols))

        # Perform bootstrap resampling for each column
        for col in range(n_cols):
            bootstrap_samples = np.random.choice(data[:, col][nan_mask[:, col]], size=(num_samples, n_rows), replace=True)
            bootstrap_means[:, col] = np.mean(bootstrap_samples, axis=1)

        # Calculate confidence interval bounds
        ci_lower = np.percentile(bootstrap_means, 100 * (alpha / 2), axis=0)
        ci_upper = np.percentile(bootstrap_means, 100 * (1 - alpha / 2), axis=0)
        
        interval = np.mean([abs(bootstrap_means - ci_lower), abs(bootstrap_means - ci_upper)], axis=0)
        
        interval = np.mean(interval, axis=0)

    else:
        raise ValueError("Invalid 'type' argument. Use either 'sem' or 'bootstrap'.")
    
    return nanmean_result, interval


def compute_null_cv_dist(avg_firing_rate, ofc_channels, min_channel_distance=20):
    """
    Compute null CV distribution from distant neuron pairs.
    
    Parameters:
    -----------
    avg_firing_rate : array, shape (n_neurons, n_timebins)
        Trial-averaged firing rates for each neuron
    ofc_channels : array, shape (n_neurons,)
        Channel number for each neuron
    min_channel_distance : int, default=20
        Minimum channel distance for null distribution pairs
    
    Returns:
    --------
    distant_pairs_cv : array
        Array of CV values from distant pairs (may contain NaN)
    """

    epsilon = 1e-6  # to avoid division by zero

    # Initialize list to store CVs from distant pairs
    distant_pairs_cv = []

    # Loop through all pairs of neurons
    for i in range(len(ofc_channels)):
        for j in range(i+1, len(ofc_channels)):  # only unique pairs
            
            # Check if they're far apart
            channel_distance = np.abs(ofc_channels[i] - ofc_channels[j])
            
            if channel_distance > min_channel_distance:
                # Get firing rates for this pair
                fr_i = avg_firing_rate[i, :]
                fr_j = avg_firing_rate[j, :]
                
                # Compute ratio at each time point
                ratio = fr_j / (fr_i + epsilon)
                
                # Compute coefficient of variation
                cv = np.std(ratio) / np.abs(np.mean(ratio))
               
                # Store it
                distant_pairs_cv.append(cv)

    # Convert to array
    distant_pairs_cv = np.array(distant_pairs_cv)
    
    return distant_pairs_cv


def find_duplicate_neurons(null_cv_dist, avg_firing_rate, ofc_channels, 
                          threshold_percentile=1, nearby_window=4):
    """
    Find potential duplicate neurons by comparing nearby pairs to null CV distribution.
    
    Parameters:
    -----------
    null_cv_dist : array
        Null distribution of CV values from distant pairs
    avg_firing_rate : array, shape (n_neurons, n_timebins)
        Trial-averaged firing rates for each neuron
    ofc_channels : array, shape (n_neurons,)
        Channel number for each neuron
    threshold_percentile : float, default=1
        Percentile threshold for flagging duplicates
    nearby_window : int, default=4
        Channel window for testing nearby pairs (±nearby_window)
    
    Returns:
    --------
    potential_duplicates : DataFrame
        DataFrame with columns: neuron_i, neuron_j, channel_i, channel_j, 
        channel_distance, cv, p_value, mean_ratio
    """
    
    epsilon = 1e-6
    
    # Compute threshold from null distribution (filter out NaNs and extremes)
    null_cv_clean = null_cv_dist[(~np.isnan(null_cv_dist)) & (null_cv_dist < 30)]
    cv_threshold = np.percentile(null_cv_clean, threshold_percentile)
    
    # Test nearby pairs
    potential_duplicates = []
    
    for i in range(len(ofc_channels)):
        channel_i = ofc_channels[i]
        
        # Define channel window
        ch_win_start = max(0, channel_i - nearby_window)
        ch_win_end = min(384, channel_i + nearby_window)
        
        # Find neurons in window (excluding self)
        nearby_neurons = np.where(
            (ofc_channels >= ch_win_start) & 
            (ofc_channels <= ch_win_end) & 
            (ofc_channels != channel_i)
        )[0]
        
        # Test each nearby pair
        for j in nearby_neurons:
            # Only test each pair once (i < j)
            if i >= j:
                continue
            
            fr_i = avg_firing_rate[i, :]
            fr_j = avg_firing_rate[j, :]
            
            # Compute ratio
            ratio = fr_j / (fr_i + epsilon)
            
            # Check if mean ratio is reasonable
            if np.abs(np.mean(ratio)) > 0.1:
                cv = np.std(ratio) / np.abs(np.mean(ratio))
                
                # Check if it's below threshold
                if cv < cv_threshold:
                    # Compute p-value
                    p_value = np.mean(null_cv_clean <= cv)
                    
                    # Store this pair
                    potential_duplicates.append({
                        'neuron_i': i,
                        'neuron_j': j,
                        'channel_i': ofc_channels[i],
                        'channel_j': ofc_channels[j],
                        'channel_distance': np.abs(ofc_channels[i] - ofc_channels[j]),
                        'cv': cv,
                        'p_value': p_value,
                        'mean_ratio': np.mean(ratio)
                    })
    
    # Convert to DataFrame
    duplicates_df = pd.DataFrame(potential_duplicates)
    
    if len(duplicates_df) > 0:
        duplicates_df = duplicates_df.sort_values('cv')
        
    return duplicates_df   


def merge_duplicates(firing_rates, potential_duplicates):
    """
    Merge duplicate neuron pairs by averaging their firing rates.
    Each neuron is involved in at most one merge (processes pairs in order of CV).
    
    Parameters:
    -----------
    firing_rates : array, shape (n_trials, n_timebins, n_neurons)
        Firing rates for all neurons
    potential_duplicates : DataFrame
        DataFrame with duplicate pairs (must have columns: neuron_i, neuron_j, cv)
    
    Returns:
    --------
    merged_firing_rates : array, shape (n_trials, n_timebins, n_neurons_merged)
        Firing rates after merging duplicates
    merge_info : dict
        Information about the merging process with keys:
        - 'n_original': original number of neurons
        - 'n_merged': number of neurons after merging
        - 'n_pairs_merged': number of pairs that were merged
        - 'neurons_removed': list of neuron indices that were removed
        - 'merge_map': dict mapping removed neurons to their merge target
    """
    
    # Copy firing rates to avoid modifying original
    merged_firing_rates = firing_rates.copy()
    
    # Sort by CV (merge most confident pairs first)
    duplicates_sorted = potential_duplicates.sort_values('cv')
    
    neurons_already_merged = set()
    neurons_merged_into = {}  # Track what got merged where
    
    print("Merging duplicate pairs...")
    for _, pair in duplicates_sorted.iterrows():
        i = int(pair['neuron_i'])
        j = int(pair['neuron_j'])
        
        # Skip if either neuron was already involved in a merge
        if i in neurons_already_merged or j in neurons_already_merged:
            continue
        
        # Average this pair across all trials and timebins
        merged_firing_rates[:, :, i] = (firing_rates[:, :, i] + firing_rates[:, :, j]) / 2
        
        # Mark both as merged
        neurons_already_merged.add(i)
        neurons_already_merged.add(j)
        neurons_merged_into[j] = i  # j was merged into i
            
    # Remove the merged-away neurons
    neurons_to_remove = sorted(list(neurons_merged_into.keys()), reverse=True)
    
    for neuron_idx in neurons_to_remove:
        merged_firing_rates = np.delete(merged_firing_rates, neuron_idx, axis=2)
    
    # Prepare merge info
    merge_info = {
        'n_original': firing_rates.shape[2],
        'n_merged': merged_firing_rates.shape[2],
        'n_pairs_merged': len(neurons_merged_into),
        'neurons_removed': neurons_to_remove,
        'merge_map': neurons_merged_into
    }
    
    return merged_firing_rates, merge_info

In [3]:
# get all the files in the directory
datadir = r'C:\Users\thome\Documents\PYTHON\OFC-CdN 3 state self control\files_for_decoder/'
data_files = glob.glob(os.path.join(datadir, '*.h5'))
file_names = [os.path.basename(file) for file in data_files]

save_dir = r'C:\Users\thome\Documents\PYTHON\OFC-CdN 3 state self control\single_neuron_summary/' 

In [4]:
h5u.list_hdf5_data(data_files[0])

Datasets in 'C:\Users\thome\Documents\PYTHON\OFC-CdN 3 state self control\files_for_decoder\D20231219_Rec05.h5':
CdN_FR
CdN_channels
CdN_lfp
CdN_locations
CdN_mean_wf
CdN_u_names
CdN_zFR
OFC_FR
OFC_channels
OFC_lfp
OFC_locations
OFC_mean_wf
OFC_u_names
OFC_zFR
bhv
chan_map
lfp_ts
ts


In [ ]:
# loop over each file
for f_ix, file_path in enumerate(data_files):
    
    print(file_names[f_ix][0:-3]) 

    # access the data for this session
    firing_rates = np.concatenate([pull_from_h5(file_path, 'CdN_zFR'), 
                                pull_from_h5(file_path, 'OFC_zFR')], axis=2)
    
    raw_firing_rates = np.concatenate([pull_from_h5(file_path, 'CdN_FR'), 
                                       pull_from_h5(file_path, 'OFC_FR')], axis=2)

    u_names = np.concatenate([pull_from_h5(file_path, 'CdN_u_names'), 
                            pull_from_h5(file_path, 'OFC_u_names')], axis=0)

    n_OFC = pull_from_h5(file_path, 'OFC_zFR').shape[2]
    n_CdN = pull_from_h5(file_path, 'CdN_zFR').shape[2]
    brain_areas = np.concatenate([np.zeros(shape=n_CdN, ), np.ones(shape=n_OFC, )]).astype(int)

    unit_waveforms = np.concatenate([pull_from_h5(file_path,'CdN_mean_wf'), 
                                     pull_from_h5(file_path,'OFC_mean_wf')], axis=0)
    
    unit_locations = np.concatenate([pull_from_h5(file_path,'CdN_locations'), 
                                 pull_from_h5(file_path,'OFC_locations')], axis=0)

    ts = pull_from_h5(file_path, 'ts')
    bhv = pd.read_hdf(file_path, key='bhv')

    if len(bhv) > len(firing_rates):
        bhv = bhv.loc[0 :len(firing_rates)-1]

    # subselect trials with a response that was correct
    trials2keep = (bhv['n_sacc'] > 0)
    bhv = bhv.loc[trials2keep]
    firing_rates = firing_rates[trials2keep, :,:]
    firing_rates = np.nan_to_num(firing_rates, nan=0)

    # get raw firing rates as well
    raw_firing_rates = raw_firing_rates[trials2keep, :,:]
    raw_firing_rates = np.nan_to_num(raw_firing_rates, nan=0)


    n_trials, n_times, n_units = np.shape(firing_rates)

    ix = (bhv['n_sacc'] < 2)

    # create indices for the states of each trial
    s1_ix = bhv['state'] == 1
    s2_ix = bhv['state'] == 2
    s3_ix = bhv['state'] == 3

    # initialize dataframe for the regression analysis
    reg_factors = pd.DataFrame()
    reg_factors['state'] = bhv['state'].copy()
    reg_factors['cue'] = bhv['state_cue'].copy()
    reg_factors['value'] = bhv['ch_val'].copy()
    reg_factors['dir'] = bhv['side'].copy()
    reg_factors['state_cue'] = bhv['state_cue'].copy()
    reg_factors['t_num'] = np.arange(0, len(bhv))

    # Define the factors (excluding intercept)
    ix = (bhv['n_sacc'] == 1)

    factors = ['state_main', 'value_main', 'state_x_value', 'direction_main','state_cue_main', 'state_x_cue']
    n_factors = len(factors)

    # Initialize arrays to store results of time-resolved regression
    t_factor_pvals = np.full((n_units, n_times, n_factors), np.nan)

    # initialize arrays to accumulate regression results for each state
    reg_betas = np.zeros((n_units, n_times, 3))
    reg_pvals = np.zeros((n_units, n_times, 3))

    # initialize array for accumulating condition mean firing rates into
    f_cond_means = np.zeros((12, n_times, n_units))
    f_cond_means_z = np.zeros((12, n_times, n_units))
    f_stateCue_cond_means = np.zeros((6, n_times, n_units))

    # loop over the neurons
    for u in tqdm(range(n_units)):
        
        # loop over each timestep
        for t in range(n_times):
            
            # Grab this neuron's firing rate at this time
            reg_factors['firing_rate'] = firing_rates[:, t, u]

            # Fit model that considers all factors
            model = smf.ols('firing_rate ~ C(state, Sum) * value + dir + state_cue + state_cue:C(state, Sum) + t_num', data=reg_factors.loc[ix]).fit()

            # compute ANOVA table from this model
            anova_table = anova_lm(model, typ=2)

            # extract p-vales for each ANOVA factor
            state_pval = anova_table.loc['C(state, Sum)', 'PR(>F)']
            value_pval = anova_table.loc['value', 'PR(>F)']
            state_x_value_pval = anova_table.loc['C(state, Sum):value', 'PR(>F)']
            direction_pval = anova_table.loc['dir', 'PR(>F)']
            state_cue_pval = anova_table.loc['state_cue', 'PR(>F)']
            state_x_cue_pval = anova_table.loc['state_cue:C(state, Sum)', 'PR(>F)']

            # save the p values
            t_factor_pvals[u, t, 0] = state_pval
            t_factor_pvals[u, t, 1] = value_pval
            t_factor_pvals[u, t, 2] = state_x_value_pval
            t_factor_pvals[u, t, 3] = direction_pval
            t_factor_pvals[u, t, 4] = state_cue_pval
            t_factor_pvals[u, t, 5] = state_x_cue_pval      

            # now assess how this neuron's firing rate scales with value in each state, separately
            for s_ix, s_mask in enumerate([s1_ix, s2_ix, s3_ix]):
                state_fr  = firing_rates[s_mask, t, u]
                state_val = bhv['ch_val'].values[s_mask]

                state_df = pd.DataFrame({'fr': state_fr, 'val': state_val})
                model = smf.ols('fr ~ val', data=state_df).fit()

                reg_betas[u, t, s_ix] = model.params['val']      
                reg_pvals[u, t, s_ix] = model.pvalues['val']   

        # now we need to get the mean firing rate of each neuron in each condition
        cond_ix = 0
        for state in [1, 2, 3]:
            for val in sorted(bhv['ch_val'].unique()):
                mask = (bhv['state'] == state) & (bhv['ch_val'] == val) & (bhv['rt'] < 500) & (bhv['n_sacc'] == 1)
                f_cond_means[cond_ix, :, u] = np.nanmean(raw_firing_rates[mask, :, u], axis=0)
                cond_ix += 1


        # now we need to get the trial-averaged firing rate of each neuron for each state-cue combination
        cond_ix = 0
        for state in [1, 2, 3]:
            for cue in [1,2]:
                mask = (bhv['state'] == state) & (bhv['state_cue'] == cue) & (bhv['rt'] < 500) & (bhv['n_sacc'] == 1)
                f_stateCue_cond_means[cond_ix, :, u] = np.nanmean(raw_firing_rates[mask, :, u], axis=0)
                cond_ix += 1




    # now save the file
    print('Saving data...')
    save_name = save_dir + file_names[f_ix][0:-3] + '_summary.h5'

    # Open an HDF5 file in write mode ('w' or 'w-' to create or truncate the file)
    with h5py.File(save_name, 'w') as file:
        # Create datasets within the HDF5 file and write data
        file.create_dataset('u_names', data = u_names)
        file.create_dataset('brain_area', data=brain_areas) 
        file.create_dataset('unit_locations', data = unit_locations) 
        file.create_dataset('unit_waveforms', data = unit_waveforms)
        file.create_dataset('ts', data=ts)  
        file.create_dataset('t_factor_pvals', data=t_factor_pvals) 
        file.create_dataset('valstate_pvals', data=reg_pvals)  
        file.create_dataset('valstate_betas', data=reg_betas)  
        file.create_dataset('stateVal_cond_means', data=f_cond_means)  
        file.create_dataset('stateCue_cond_means', data=f_stateCue_cond_means) 
        
    print('Data saved \n')

print('All files processed :]')

    
            
    

D20231219_Rec05


100%|██████████| 787/787 [27:30<00:00,  2.10s/it]


Saving data...
Data saved 

D20231221_Rec06


100%|██████████| 606/606 [15:13<00:00,  1.51s/it]


Saving data...
Data saved 

D20231224_Rec07


...

Saving data...
Data saved 

K20240712_Rec08


100%|██████████| 520/520 [11:12<00:00,  1.29s/it]


Saving data...
Data saved 

K20240715_Rec09


100%|██████████| 606/606 [12:17<00:00,  1.22s/it]

Saving data...
Data saved 

All files processed :]
